# 06 — Power BI Data Preparation

## UK Data Analyst Job Market Intelligence

This notebook prepares the analytical data model used by the Power BI
dashboard.

The objective is to transform the cleaned job-market dataset into
BI-friendly fact, dimension and bridge tables that support interactive
analysis without duplicating vacancy records.

The Power BI model will support analysis across:

- job family;
- geography;
- salary;
- seniority;
- contract structure;
- advertiser;
- technology mentions;
- analytical capabilities;
- search-query coverage.

In [29]:
import os
import pandas as pd
import numpy as np
import re
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
PROJECT_PATH = "/content/drive/MyDrive/UK_Data_Analyst_Market"

PROCESSED_PATH = f"{PROJECT_PATH}/data/processed"
POWERBI_PATH = f"{PROJECT_PATH}/data/powerbi"

os.makedirs(
    POWERBI_PATH,
    exist_ok=True
)

print(POWERBI_PATH)

/content/drive/MyDrive/UK_Data_Analyst_Market/data/powerbi


In [31]:
core_jobs = pd.read_csv(
    f"{PROCESSED_PATH}/uk_core_analytics_jobs.csv"
)

job_search_terms = pd.read_csv(
    f"{PROCESSED_PATH}/job_search_terms.csv"
)

print("Core jobs:", len(core_jobs))
print("Search-term relations:", len(job_search_terms))

Core jobs: 244
Search-term relations: 532


## 1. FactJobs

`FactJobs` is the central fact table of the Power BI model.

Each row represents one unique Core Analytics vacancy. Derived fields used for
dashboard filtering and segmentation are created here, while job-level
uniqueness is preserved.

In [32]:
core_jobs["job_id"] = (
    core_jobs["job_id"]
    .astype("string")
)

job_search_terms["job_id"] = (
    job_search_terms["job_id"]
    .astype("string")
)

core_jobs["created"] = pd.to_datetime(
    core_jobs["created"],
    utc=True,
    errors="coerce"
)

In [33]:
def classify_geo_scope(location):

    if pd.isna(location):
        return "Unknown"

    location = str(location).strip()

    if "london" in location.lower():
        return "London"

    if location.lower() == "uk":
        return "UK-wide / Unspecified"

    return "Rest of UK"


core_jobs["geo_scope"] = (
    core_jobs["location"]
    .apply(classify_geo_scope)
)

In [34]:
core_jobs["geo_scope"].value_counts()

,count
geo_scope,
Rest of UK,140
London,75
UK-wide / Unspecified,29


In [35]:
def classify_salary_band(salary):

    if pd.isna(salary):
        return "Unknown"

    if salary < 30000:
        return "Under £30k"

    if salary < 40000:
        return "£30k–£39,999"

    if salary < 50000:
        return "£40k–£49,999"

    if salary < 60000:
        return "£50k–£59,999"

    if salary < 80000:
        return "£60k–£79,999"

    if salary < 100000:
        return "£80k–£99,999"

    return "£100k+"


core_jobs["salary_band"] = (
    core_jobs["salary_midpoint"]
    .apply(classify_salary_band)
)

In [36]:
fact_jobs_columns = [
    "job_id",
    "title",
    "company",
    "location",
    "job_family",
    "seniority",
    "created",
    "salary_min",
    "salary_max",
    "salary_midpoint",
    "salary_is_predicted",
    "salary_source",
    "salary_band",
    "geo_scope",
    "contract_type",
    "contract_time"
]

FactJobs = (
    core_jobs[
        fact_jobs_columns
    ]
    .copy()
    .reset_index(drop=True)
)

In [37]:
print("FactJobs rows:", len(FactJobs))
print(
    "Unique job IDs:",
    FactJobs["job_id"].nunique()
)

print(
    "Duplicate job IDs:",
    FactJobs["job_id"].duplicated().sum()
)

FactJobs rows: 244
Unique job IDs: 244
Duplicate job IDs: 0


In [38]:
FactJobs.head()

,job_id,title,company,location,job_family,seniority,created,salary_min,salary_max,salary_midpoint,salary_is_predicted,salary_source,salary_band,geo_scope,contract_type,contract_time
0,5840324082,Lead Data Analyst/Project Controller,BAE Systems,"Padiham, Burnley",Data Analyst,Lead / Principal,2026-08-13 15:06:21+00:00,53889.32,53889.32,53889.32,1,Adzuna predicted,"£50k–£59,999",Rest of UK,NaN,NaN
1,5840324089,Lead Data Analyst/Project Controller,BAE Systems,"Penwortham, Preston",Data Analyst,Lead / Principal,2026-08-13 15:06:21+00:00,53416.18,53416.18,53416.18,1,Adzuna predicted,"£50k–£59,999",Rest of UK,NaN,NaN
2,5840324100,Lead Data Analyst/Project Controller,BAE Systems,"Samlesbury, Preston",Data Analyst,Lead / Principal,2026-08-13 15:06:21+00:00,63011.66,63011.66,63011.66,1,Adzuna predicted,"£60k–£79,999",Rest of UK,NaN,NaN
3,5835920078,Data Analyst,VANRATH,"Antrim, County Antrim",Data Analyst,Unspecified,2026-08-10 20:53:51+00:00,40000.00,40000.00,40000.00,0,Advertised / non-predicted,"£40k–£49,999",Rest of UK,NaN,NaN
4,5829131501,Data Analyst,MCS Group,"Banbridge, County Down",Data Analyst,Unspecified,2026-08-05 15:10:55+00:00,30000.00,30000.00,30000.00,0,Advertised / non-predicted,"£30k–£39,999",Rest of UK,NaN,NaN


In [39]:
FactJobs["created"] = (
    FactJobs["created"]
    .dt.tz_convert(None)
)

In [40]:
FactJobs.to_csv(
    f"{POWERBI_PATH}/FactJobs.csv",
    index=False
)

print("FactJobs exported.")

FactJobs exported.


In [41]:
print(
    os.path.exists(
        f"{POWERBI_PATH}/FactJobs.csv"
    )
)

True


## 2. Technology Model

Technology mentions form a many-to-many analytical relationship:

- one vacancy may mention multiple technologies;
- one technology may appear across multiple vacancies.

To preserve one row per vacancy in `FactJobs`, technology mentions are stored
in a separate bridge table connected to a technology dimension.

Only technologies explicitly detected in the available Adzuna description
snippets are represented.

In [42]:
technology_patterns = {

    # Programming / query
    "SQL": r"\bsql\b",
    "Python": r"\bpython\b",

    # Spreadsheet
    "Excel": r"\bexcel\b|microsoft excel",

    # BI / visualisation
    "Power BI": r"\bpower\s*bi\b|powerbi",
    "Tableau": r"\btableau\b",
    "Looker": r"\blooker\b",
    "Qlik": r"\bqlik(?:view|sense)?\b",

    # Data platforms / databases
    "Snowflake": r"\bsnowflake\b",
    "BigQuery": r"\bbigquery\b|big query",
    "Databricks": r"\bdatabricks\b",
    "SQL Server": r"\bsql\s+server\b",

    # Cloud
    "Azure": r"\bazure\b",
    "AWS": r"\baws\b|amazon web services",
    "Google Cloud": r"\bgoogle cloud\b|\bgcp\b",

    # Transformation
    "ETL": r"\betl\b",
    "dbt": r"\bdbt\b",
    "Power Query": r"\bpower\s+query\b",

    # Enterprise platforms
    "Oracle": r"\boracle\b",
    "SAP": r"\bsap\b",
    "Salesforce": r"\bsalesforce\b",

    # Delivery / collaboration
    "Jira": r"\bjira\b",

    # Version control
    "Git": r"\bgit\b|github|gitlab"
}

In [43]:
def extract_technologies(text):

    if pd.isna(text):
        return []

    found = []

    for technology, pattern in technology_patterns.items():

        if re.search(
            pattern,
            str(text),
            flags=re.IGNORECASE
        ):
            found.append(technology)

    return found


core_jobs["technologies_found"] = (
    core_jobs["description"]
    .apply(extract_technologies)
)

In [44]:
core_jobs[
    [
        "job_id",
        "title",
        "technologies_found"
    ]
].head(20)

,job_id,title,technologies_found
0,5840324082,Lead Data Analyst/Project Controller,[]
1,5840324089,Lead Data Analyst/Project Controller,[]
2,5840324100,Lead Data Analyst/Project Controller,[]
3,5835920078,Data Analyst,[]
4,5829131501,Data Analyst,[]
5,5823507727,Data Analyst,[]
6,5818756002,Master Data Analyst,[]
7,5818755973,Data Analyst - Manufacturing,[]
8,5840658196,Lead Data Analyst/Project Controller,[]
9,5812354048,Data Analyst,[]


In [45]:
BridgeJobTechnology = (
    core_jobs[
        [
            "job_id",
            "technologies_found"
        ]
    ]
    .explode("technologies_found")
    .rename(
        columns={
            "technologies_found": "technology"
        }
    )
)

BridgeJobTechnology = (
    BridgeJobTechnology[
        BridgeJobTechnology["technology"].notna()
    ]
    .drop_duplicates(
        subset=[
            "job_id",
            "technology"
        ]
    )
    .reset_index(drop=True)
)

In [46]:
print(
    "Bridge rows:",
    len(BridgeJobTechnology)
)

print(
    "Jobs with technology mentions:",
    BridgeJobTechnology["job_id"].nunique()
)

print(
    "Unique technologies:",
    BridgeJobTechnology["technology"].nunique()
)

print(
    "Duplicate job-technology pairs:",
    BridgeJobTechnology[
        ["job_id", "technology"]
    ].duplicated().sum()
)

Bridge rows: 66
Jobs with technology mentions: 38
Unique technologies: 14
Duplicate job-technology pairs: 0


In [47]:
BridgeJobTechnology.head(20)

,job_id,technology
0,5840637955,SQL
1,5838674911,Power BI
2,5835867417,SQL
3,5835867417,Python
4,5835867417,Power BI
5,5835868209,SQL
6,5835868209,Excel
7,5835868209,Power BI
8,5810846111,Excel
9,5810846111,Power BI


In [48]:
technology_validation = (
    BridgeJobTechnology["technology"]
    .value_counts()
    .rename_axis("technology")
    .reset_index(name="vacancies")
)

technology_validation

,technology,vacancies
0,Power BI,24
1,SQL,14
2,Excel,11
3,Python,4
4,Tableau,3
5,SQL Server,2
6,Snowflake,1
7,Salesforce,1
8,Azure,1
9,Power Query,1


In [49]:
technology_categories = {

    "SQL": "Programming / Query",
    "Python": "Programming",

    "Excel": "Spreadsheet",

    "Power BI": "BI / Visualisation",
    "Tableau": "BI / Visualisation",
    "Looker": "BI / Visualisation",
    "Qlik": "BI / Visualisation",

    "Snowflake": "Data Platform",
    "BigQuery": "Data Platform",
    "Databricks": "Data Platform",

    "SQL Server": "Database",

    "Azure": "Cloud",
    "AWS": "Cloud",
    "Google Cloud": "Cloud",

    "ETL": "Data Engineering",
    "dbt": "Data Engineering",
    "Power Query": "Data Transformation",

    "Oracle": "Enterprise / Database",
    "SAP": "Enterprise Platform",
    "Salesforce": "Enterprise Platform",

    "Jira": "Collaboration / Delivery",

    "Git": "Version Control"
}

In [50]:
DimTechnology = (
    BridgeJobTechnology[
        ["technology"]
    ]
    .drop_duplicates()
    .sort_values("technology")
    .reset_index(drop=True)
)

In [51]:
DimTechnology["technology_category"] = (
    DimTechnology["technology"]
    .map(technology_categories)
    .fillna("Other")
)

In [52]:
DimTechnology

,technology,technology_category
0,Azure,Cloud
1,ETL,Data Engineering
2,Excel,Spreadsheet
3,Jira,Collaboration / Delivery
4,Oracle,Enterprise / Database
5,Power BI,BI / Visualisation
6,Power Query,Data Transformation
7,Python,Programming
8,SAP,Enterprise Platform
9,SQL,Programming / Query


In [54]:
print(DimTechnology.columns.tolist())

['technology', 'technology_category']


In [56]:
DimTechnology = (
    BridgeJobTechnology[
        ["technology"]
    ]
    .drop_duplicates()
    .sort_values("technology")
    .reset_index(drop=True)
)

DimTechnology["technology_id"] = (
    range(1, len(DimTechnology) + 1)
)

DimTechnology["technology_category"] = (
    DimTechnology["technology"]
    .map(technology_categories)
    .fillna("Other")
)

DimTechnology = DimTechnology[
    [
        "technology_id",
        "technology",
        "technology_category"
    ]
]

DimTechnology

,technology_id,technology,technology_category
0,1,Azure,Cloud
1,2,ETL,Data Engineering
2,3,Excel,Spreadsheet
3,4,Jira,Collaboration / Delivery
4,5,Oracle,Enterprise / Database
5,6,Power BI,BI / Visualisation
6,7,Power Query,Data Transformation
7,8,Python,Programming
8,9,SAP,Enterprise Platform
9,10,SQL,Programming / Query


In [58]:
print(DimTechnology.columns.tolist())
print("Technologies:", len(DimTechnology))
print(
    "Unique technology IDs:",
    DimTechnology["technology_id"].nunique()
)

['technology_id', 'technology', 'technology_category']
Technologies: 14
Unique technology IDs: 14


In [59]:
BridgeJobTechnology = (
    BridgeJobTechnology
    .merge(
        DimTechnology[
            [
                "technology_id",
                "technology"
            ]
        ],
        on="technology",
        how="left",
        validate="many_to_one"
    )
)

In [61]:
BridgeJobTechnology.head(20)

,job_id,technology,technology_id_x,technology_id_y
0,5840637955,SQL,10,10
1,5838674911,Power BI,6,6
2,5835867417,SQL,10,10
3,5835867417,Python,8,8
4,5835867417,Power BI,6,6
5,5835868209,SQL,10,10
6,5835868209,Excel,3,3
7,5835868209,Power BI,6,6
8,5810846111,Excel,3,3
9,5810846111,Power BI,6,6


In [63]:
print("Bridge columns:", BridgeJobTechnology.columns.tolist())
print("Dim columns:", DimTechnology.columns.tolist())

Bridge columns: ['job_id', 'technology', 'technology_id_x', 'technology_id_y']
Dim columns: ['technology_id', 'technology', 'technology_category']


In [64]:
# --------------------------------------------------
# Rebuild BridgeJobTechnology from source data
# --------------------------------------------------

BridgeJobTechnology = (
    core_jobs[
        [
            "job_id",
            "technologies_found"
        ]
    ]
    .explode("technologies_found")
    .rename(
        columns={
            "technologies_found": "technology"
        }
    )
)

BridgeJobTechnology = (
    BridgeJobTechnology[
        BridgeJobTechnology["technology"].notna()
    ]
    .drop_duplicates(
        subset=[
            "job_id",
            "technology"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Bridge before dimension merge:",
    BridgeJobTechnology.shape
)

print(
    BridgeJobTechnology.columns.tolist()
)

Bridge before dimension merge: (66, 2)
['job_id', 'technology']


In [65]:
DimTechnology = (
    BridgeJobTechnology[
        ["technology"]
    ]
    .drop_duplicates()
    .sort_values("technology")
    .reset_index(drop=True)
)

DimTechnology["technology_id"] = (
    range(1, len(DimTechnology) + 1)
)

DimTechnology["technology_category"] = (
    DimTechnology["technology"]
    .map(technology_categories)
    .fillna("Other")
)

DimTechnology = DimTechnology[
    [
        "technology_id",
        "technology",
        "technology_category"
    ]
]

DimTechnology

,technology_id,technology,technology_category
0,1,Azure,Cloud
1,2,ETL,Data Engineering
2,3,Excel,Spreadsheet
3,4,Jira,Collaboration / Delivery
4,5,Oracle,Enterprise / Database
5,6,Power BI,BI / Visualisation
6,7,Power Query,Data Transformation
7,8,Python,Programming
8,9,SAP,Enterprise Platform
9,10,SQL,Programming / Query


In [66]:
print(
    "DimTechnology columns:",
    DimTechnology.columns.tolist()
)

print(
    "Unique IDs:",
    DimTechnology["technology_id"].nunique()
)

print(
    "Rows:",
    len(DimTechnology)
)

DimTechnology columns: ['technology_id', 'technology', 'technology_category']
Unique IDs: 14
Rows: 14


In [67]:
BridgeJobTechnology = (
    BridgeJobTechnology
    .merge(
        DimTechnology[
            [
                "technology_id",
                "technology"
            ]
        ],
        on="technology",
        how="left",
        validate="many_to_one"
    )
)

BridgeJobTechnology.head(20)

,job_id,technology,technology_id
0,5840637955,SQL,10
1,5838674911,Power BI,6
2,5835867417,SQL,10
3,5835867417,Python,8
4,5835867417,Power BI,6
5,5835868209,SQL,10
6,5835868209,Excel,3
7,5835868209,Power BI,6
8,5810846111,Excel,3
9,5810846111,Power BI,6


In [68]:
print(
    BridgeJobTechnology.columns.tolist()
)

['job_id', 'technology', 'technology_id']


In [69]:
print(
    "Missing technology IDs:",
    BridgeJobTechnology["technology_id"]
    .isna()
    .sum()
)

Missing technology IDs: 0


In [70]:
print(
    "Duplicate job-technology pairs:",
    BridgeJobTechnology[
        [
            "job_id",
            "technology_id"
        ]
    ]
    .duplicated()
    .sum()
)

Duplicate job-technology pairs: 0


In [71]:
BridgeJobTechnology = (
    BridgeJobTechnology[
        [
            "job_id",
            "technology_id"
        ]
    ]
    .copy()
)

In [72]:
print("Bridge rows:", len(BridgeJobTechnology))

print(
    "Missing technology IDs:",
    BridgeJobTechnology["technology_id"]
    .isna()
    .sum()
)

print(
    "Duplicate bridge pairs:",
    BridgeJobTechnology[
        [
            "job_id",
            "technology_id"
        ]
    ]
    .duplicated()
    .sum()
)

print(
    "Dimension rows:",
    len(DimTechnology)
)

Bridge rows: 66
Missing technology IDs: 0
Duplicate bridge pairs: 0
Dimension rows: 14


In [73]:
BridgeJobTechnology.to_csv(
    f"{POWERBI_PATH}/BridgeJobTechnology.csv",
    index=False
)

DimTechnology.to_csv(
    f"{POWERBI_PATH}/DimTechnology.csv",
    index=False
)

print("Technology tables exported.")

Technology tables exported.


## 3. Capability Model

Analytical capabilities are modelled separately from technologies.

This distinction prevents concepts such as Reporting, Dashboarding and Data
Analysis from being treated as equivalent to specific tools such as SQL,
Power BI or Python.

A bridge table preserves the many-to-many relationship between vacancies and
capabilities while maintaining one row per vacancy in `FactJobs`.

In [74]:
capability_patterns = {

    "Data Analysis":
        r"\bdata analysis\b|\banalyse data\b|\banalyze data\b",

    "Reporting":
        r"\breporting\b|\breport development\b",

    "Dashboarding":
        r"\bdashboards?\b|\bdashboard development\b",

    "Data Visualisation":
        r"data visuali[sz]ation|visuali[sz]e data",

    "Data Modelling":
        r"\bdata modelling\b|\bdata modeling\b",

    "Data Quality":
        r"\bdata quality\b",

    "Data Governance":
        r"\bdata governance\b",

    "Statistics":
        r"\bstatistics?\b|\bstatistical analysis\b",

    "KPI":
        r"\bkpis?\b|key performance indicators?",

    "Forecasting":
        r"\bforecasting\b|\bforecast\b",

    "Stakeholder Management":
        r"\bstakeholder management\b|"
        r"\bstakeholder engagement\b",

    "Requirements Gathering":
        r"\brequirements gathering\b|"
        r"\bgathering requirements\b",

    "Data Cleaning":
        r"\bdata cleaning\b|"
        r"\bcleaning data\b",

    "Data Transformation":
        r"\bdata transformation\b|"
        r"\btransforming data\b",

    "Data Storytelling":
        r"\bdata storytelling\b|"
        r"\bstorytelling with data\b"
}

In [75]:
def extract_capabilities(text):

    if pd.isna(text):
        return []

    found = []

    for capability, pattern in capability_patterns.items():

        if re.search(
            pattern,
            str(text),
            flags=re.IGNORECASE
        ):
            found.append(capability)

    return found

In [76]:
core_jobs["capabilities_found"] = (
    core_jobs["description"]
    .apply(extract_capabilities)
)

In [77]:
core_jobs[
    [
        "job_id",
        "title",
        "capabilities_found"
    ]
].head(20)

,job_id,title,capabilities_found
0,5840324082,Lead Data Analyst/Project Controller,[]
1,5840324089,Lead Data Analyst/Project Controller,[]
2,5840324100,Lead Data Analyst/Project Controller,[]
3,5835920078,Data Analyst,[]
4,5829131501,Data Analyst,[Reporting]
5,5823507727,Data Analyst,[]
6,5818756002,Master Data Analyst,[]
7,5818755973,Data Analyst - Manufacturing,[Reporting]
8,5840658196,Lead Data Analyst/Project Controller,[]
9,5812354048,Data Analyst,[Reporting]


In [78]:
BridgeJobCapability = (
    core_jobs[
        [
            "job_id",
            "capabilities_found"
        ]
    ]
    .explode("capabilities_found")
    .rename(
        columns={
            "capabilities_found": "capability"
        }
    )
)

BridgeJobCapability = (
    BridgeJobCapability[
        BridgeJobCapability["capability"].notna()
    ]
    .drop_duplicates(
        subset=[
            "job_id",
            "capability"
        ]
    )
    .reset_index(drop=True)
)

In [79]:
capability_validation = (
    BridgeJobCapability["capability"]
    .value_counts()
    .rename_axis("capability")
    .reset_index(name="vacancies")
)

capability_validation

,capability,vacancies
0,Reporting,98
1,Dashboarding,32
2,Data Analysis,20
3,Data Transformation,8
4,Data Visualisation,6
5,KPI,5
6,Data Quality,5
7,Data Modelling,3
8,Forecasting,1


In [80]:
capability_categories = {

    "Data Analysis": "Analysis",
    "Statistics": "Analysis",
    "Forecasting": "Analysis",

    "Reporting": "Reporting & Communication",
    "Dashboarding": "Reporting & Communication",
    "Data Visualisation": "Reporting & Communication",
    "Data Storytelling": "Reporting & Communication",
    "KPI": "Reporting & Communication",

    "Data Modelling": "Data Management",
    "Data Quality": "Data Management",
    "Data Governance": "Data Management",
    "Data Cleaning": "Data Management",
    "Data Transformation": "Data Management",

    "Stakeholder Management": "Business Collaboration",
    "Requirements Gathering": "Business Collaboration"
}

In [81]:
DimCapability = (
    BridgeJobCapability[
        ["capability"]
    ]
    .drop_duplicates()
    .sort_values("capability")
    .reset_index(drop=True)
)

DimCapability["capability_id"] = (
    range(1, len(DimCapability) + 1)
)

DimCapability["capability_category"] = (
    DimCapability["capability"]
    .map(capability_categories)
    .fillna("Other")
)

DimCapability = DimCapability[
    [
        "capability_id",
        "capability",
        "capability_category"
    ]
]

DimCapability

,capability_id,capability,capability_category
0,1,Dashboarding,Reporting & Communication
1,2,Data Analysis,Analysis
2,3,Data Modelling,Data Management
3,4,Data Quality,Data Management
4,5,Data Transformation,Data Management
5,6,Data Visualisation,Reporting & Communication
6,7,Forecasting,Analysis
7,8,KPI,Reporting & Communication
8,9,Reporting,Reporting & Communication


In [82]:
BridgeJobCapability = (
    BridgeJobCapability
    .merge(
        DimCapability[
            [
                "capability_id",
                "capability"
            ]
        ],
        on="capability",
        how="left",
        validate="many_to_one"
    )
)

In [83]:
print(
    "Missing capability IDs:",
    BridgeJobCapability["capability_id"]
    .isna()
    .sum()
)

print(
    "Duplicate job-capability pairs:",
    BridgeJobCapability[
        [
            "job_id",
            "capability_id"
        ]
    ]
    .duplicated()
    .sum()
)

Missing capability IDs: 0
Duplicate job-capability pairs: 0


In [84]:
BridgeJobCapability = (
    BridgeJobCapability[
        [
            "job_id",
            "capability_id"
        ]
    ]
    .copy()
)

In [85]:
print(
    "Bridge rows:",
    len(BridgeJobCapability)
)

print(
    "Jobs with capability mentions:",
    BridgeJobCapability["job_id"].nunique()
)

print(
    "Dimension rows:",
    len(DimCapability)
)

print(
    "Duplicate bridge pairs:",
    BridgeJobCapability[
        [
            "job_id",
            "capability_id"
        ]
    ]
    .duplicated()
    .sum()
)

Bridge rows: 178
Jobs with capability mentions: 127
Dimension rows: 9
Duplicate bridge pairs: 0


In [86]:
BridgeJobCapability.to_csv(
    f"{POWERBI_PATH}/BridgeJobCapability.csv",
    index=False
)

DimCapability.to_csv(
    f"{POWERBI_PATH}/DimCapability.csv",
    index=False
)

print("Capability tables exported.")

Capability tables exported.


## 4. Date Dimension

A dedicated date dimension supports time-based filtering and aggregation in
Power BI.

The dimension is derived from vacancy posting dates and provides standard
calendar attributes such as year, month and weekday.

In [87]:
core_jobs["created"] = pd.to_datetime(
    core_jobs["created"],
    errors="coerce"
)

In [88]:
core_jobs["created_date"] = (
    core_jobs["created"]
    .dt.normalize()
)

In [89]:
date_min = core_jobs["created_date"].min()
date_max = core_jobs["created_date"].max()

date_range = pd.date_range(
    start=date_min,
    end=date_max,
    freq="D"
)

DimDate = pd.DataFrame({
    "date": date_range
})

In [90]:
DimDate["year"] = DimDate["date"].dt.year
DimDate["month_number"] = DimDate["date"].dt.month
DimDate["month_name"] = DimDate["date"].dt.month_name()
DimDate["year_month"] = DimDate["date"].dt.strftime("%Y-%m")
DimDate["quarter"] = (
    "Q"
    + DimDate["date"].dt.quarter.astype(str)
)
DimDate["day_of_week_number"] = DimDate["date"].dt.dayofweek + 1
DimDate["day_of_week"] = DimDate["date"].dt.day_name()

In [91]:
FactJobs["created_date"] = (
    pd.to_datetime(FactJobs["created"])
    .dt.normalize()
)

In [92]:
print("Date dimension rows:", len(DimDate))
print("Minimum date:", DimDate["date"].min())
print("Maximum date:", DimDate["date"].max())

print(
    "FactJobs dates missing from DimDate:",
    (~FactJobs["created_date"].isin(DimDate["date"])).sum()
)

Date dimension rows: 1197
Minimum date: 2023-05-11 00:00:00+00:00
Maximum date: 2026-08-19 00:00:00+00:00
FactJobs dates missing from DimDate: 244


In [93]:
# --------------------------------------------------
# Standardise dates as timezone-naive
# --------------------------------------------------

FactJobs["created"] = (
    pd.to_datetime(
        FactJobs["created"],
        utc=True,
        errors="coerce"
    )
    .dt.tz_convert(None)
)

FactJobs["created_date"] = (
    FactJobs["created"]
    .dt.normalize()
)

# Build date dimension directly from FactJobs
date_min = FactJobs["created_date"].min()
date_max = FactJobs["created_date"].max()

date_range = pd.date_range(
    start=date_min,
    end=date_max,
    freq="D"
)

DimDate = pd.DataFrame({
    "date": date_range
})

DimDate["year"] = DimDate["date"].dt.year
DimDate["month_number"] = DimDate["date"].dt.month
DimDate["month_name"] = DimDate["date"].dt.month_name()
DimDate["year_month"] = DimDate["date"].dt.strftime("%Y-%m")
DimDate["quarter"] = (
    "Q" + DimDate["date"].dt.quarter.astype(str)
)
DimDate["day_of_week_number"] = (
    DimDate["date"].dt.dayofweek + 1
)
DimDate["day_of_week"] = (
    DimDate["date"].dt.day_name()
)

In [94]:
print("FactJobs date dtype:", FactJobs["created_date"].dtype)
print("DimDate date dtype:", DimDate["date"].dtype)

FactJobs date dtype: datetime64[ns]
DimDate date dtype: datetime64[ns]


In [95]:
print("Date dimension rows:", len(DimDate))
print("Minimum date:", DimDate["date"].min())
print("Maximum date:", DimDate["date"].max())

print(
    "FactJobs dates missing from DimDate:",
    (~FactJobs["created_date"].isin(DimDate["date"])).sum()
)

Date dimension rows: 1197
Minimum date: 2023-05-11 00:00:00
Maximum date: 2026-08-19 00:00:00
FactJobs dates missing from DimDate: 0


In [96]:
FactJobs.to_csv(
    f"{POWERBI_PATH}/FactJobs.csv",
    index=False
)

DimDate.to_csv(
    f"{POWERBI_PATH}/DimDate.csv",
    index=False
)

print("FactJobs and DimDate re-exported.")

FactJobs and DimDate re-exported.


Resume the IntelliShop wireframe task now. Do not run Code Connect yet.

Code Connect is out of scope for this stage because we are still working on low-fidelity UX wireframes and have not yet finalised or published production UI components, nor mapped them to implementation components in the codebase.

Please continue from the previous IntelliShop brief.

Figma file:
https://www.figma.com/design/VZj2AU8kOJqYvyjp8AvWQa/IntelliShop-Product-Redesign-2026

First inspect:

03 — Product Reframing + IA
04 — Low-Fidelity Specification

Before modifying the canvas, report back these five points from the existing documentation:

Approved navigation model
Core product object
Role of AI
Purpose of Shopping Mode
Role of Pantry

Do not reinterpret or redesign these decisions.

Once you confirm them, proceed directly with creating:

05 — Low-Fidelity Wireframes

Follow the full low-fidelity wireframe brief I provided previously.

Build the complete editable vertical slice:

Home
→ List Detail
→ Add Items
→ Optimisation Review
→ Recommended Store
→ Shopping Mode
→ Finish Shopping
→ Pantry Update
→ Pantry

Also create Compare Stores as the secondary branch from Recommended Store.

Important:

390 px mobile frames
grayscale only
native editable Figma objects
Auto Layout
reusable low-fi components
no final branding
no final colour system
no final typography decisions
no decorative illustration
no high-fidelity polish
do not use Code Connect

The purpose of this stage is to validate information hierarchy, navigation, interaction, system feedback, cognitive load and task completion.

When finished, report:

page created
screens created
reusable components created
any genuine UX ambiguity that requires a product decision

Do not proceed to high fidelity.

In [97]:
BridgeJobSearchTerm = (
    job_search_terms[
        job_search_terms["job_id"].isin(
            FactJobs["job_id"]
        )
    ]
    .copy()
)

In [98]:
print(
    "Bridge search rows:",
    len(BridgeJobSearchTerm)
)

print(
    "Unique jobs:",
    BridgeJobSearchTerm["job_id"].nunique()
)

print(
    "Unique search terms:",
    BridgeJobSearchTerm["search_term"].nunique()
)

Bridge search rows: 258
Unique jobs: 244
Unique search terms: 3


In [99]:
DimSearchTerm = (
    BridgeJobSearchTerm[
        ["search_term"]
    ]
    .drop_duplicates()
    .sort_values("search_term")
    .reset_index(drop=True)
)

DimSearchTerm["search_term_id"] = (
    range(1, len(DimSearchTerm) + 1)
)

DimSearchTerm = DimSearchTerm[
    [
        "search_term_id",
        "search_term"
    ]
]

DimSearchTerm

,search_term_id,search_term
0,1,business intelligence analyst
1,2,data analyst
2,3,reporting analyst


In [100]:
BridgeJobSearchTerm = (
    BridgeJobSearchTerm
    .merge(
        DimSearchTerm,
        on="search_term",
        how="left",
        validate="many_to_one"
    )
)

In [101]:
print(
    "Missing search term IDs:",
    BridgeJobSearchTerm["search_term_id"]
    .isna()
    .sum()
)

print(
    "Duplicate job-search pairs:",
    BridgeJobSearchTerm[
        [
            "job_id",
            "search_term_id"
        ]
    ]
    .duplicated()
    .sum()
)

Missing search term IDs: 0
Duplicate job-search pairs: 0


In [102]:
BridgeJobSearchTerm = (
    BridgeJobSearchTerm[
        [
            "job_id",
            "search_term_id"
        ]
    ]
    .copy()
)

In [103]:
BridgeJobSearchTerm.to_csv(
    f"{POWERBI_PATH}/BridgeJobSearchTerm.csv",
    index=False
)

DimSearchTerm.to_csv(
    f"{POWERBI_PATH}/DimSearchTerm.csv",
    index=False
)

print("Search-term tables exported.")

Search-term tables exported.


The Master Handoff file is actually on disk. Let me read it directly.

Viewed 4 files · 1 note

I have the full Master Handoff content now. Let me perform the complete reconciliation — updating the Master Handoff, the Portfolio Materials Index, and producing the actual completion audit.

Edited 8 files, ran 3 commands, created 3 files · 6 notes

Landing is closed in full (40 now CLOSED/APPROVED), and the Phase 6 completion audit is done.

One thing worth flagging directly: I went and read the actual LUFI-PHASE6-MASTER-HANDOFF.md file directly off disk rather than working from the summary in memory — it's 376 lines and was genuinely out of date (last touched mid-Morning-Moment, before that workstream's Mid-Fi/High-Fi and all of Landing existed). I updated it in place rather than treating it as settled. I also found a real gap while doing this: LUFI-PHASE6-OPEN-DEPENDENCIES.md is referenced throughout as a companion file but was never actually uploaded to this container — only the Handoff and Canonical File Map were. Its content isn't lost (it lives in-line in the Handoff's Sections 5–6, which are current), but I flagged this as an honest synchronization gap rather than fabricating a reconciliation against a file I don't have.

Confirmed: 11 of 13 fidelity-staged workstreams are CLOSED/APPROVED across all four stages. Only Final Design System Presentation and Portfolio Production remain unstarted, as intended.

On the portfolio index — beyond adding Landing and updating Morning Moment's framing, two things worth naming:

I downgraded plans/19 and first-meeting/18 — not deleted, but marked as weaker than their original listing suggested, since later human-approved work (plans/29, plans/36) covers the same ground with stronger evidence, and first-meeting/18 turns out to be a recorded idea with no actual built artifact behind it.
I noticed something the index hadn't named explicitly: five candidates across five independent workstreams (Plans, Transition Recognition, You & LUFI, Morning Moment, Landing) are all the same underlying pattern — an older document or live surface asserting relationship depth in a way that conflicts with locked principles, caught and corrected. I recorded this as a throughline note, not a sixth candidate, so nothing gets double-counted — but if portfolio production wants a "systems thinking" narrative, this is the strongest version of it available.

Ranked candidates, if useful: Landing's full relationship-ladder arc now edges out even Plans' Companion Evolution story for the top slot, specifically because it was re-verified against recurrence three separate times rather than corrected once.

Nothing was found in conflict during reconciliation. No workstream needs reopening.

What remains before Portfolio Production: really just the Final Design System Presentation (the one item still open on the Master Brief's own numbered list) and your go-ahead — the engineering/business dependencies are explicitly not blockers, consistent with how this project has operated throughout.

In [104]:
print("===== FACT TABLE =====")

print("FactJobs rows:", len(FactJobs))
print("Unique job IDs:", FactJobs["job_id"].nunique())
print("Duplicate job IDs:", FactJobs["job_id"].duplicated().sum())


print("\n===== TECHNOLOGY =====")

print("DimTechnology rows:", len(DimTechnology))
print(
    "Unique technology IDs:",
    DimTechnology["technology_id"].nunique()
)
print(
    "BridgeJobTechnology rows:",
    len(BridgeJobTechnology)
)


print("\n===== CAPABILITY =====")

print("DimCapability rows:", len(DimCapability))
print(
    "Unique capability IDs:",
    DimCapability["capability_id"].nunique()
)
print(
    "BridgeJobCapability rows:",
    len(BridgeJobCapability)
)


print("\n===== SEARCH TERMS =====")

print("DimSearchTerm rows:", len(DimSearchTerm))
print(
    "Unique search-term IDs:",
    DimSearchTerm["search_term_id"].nunique()
)
print(
    "BridgeJobSearchTerm rows:",
    len(BridgeJobSearchTerm)
)


print("\n===== DATE =====")

print("DimDate rows:", len(DimDate))
print(
    "Fact dates missing from DimDate:",
    (~FactJobs["created_date"].isin(DimDate["date"])).sum()
)

===== FACT TABLE =====
FactJobs rows: 244
Unique job IDs: 244
Duplicate job IDs: 0

===== TECHNOLOGY =====
DimTechnology rows: 14
Unique technology IDs: 14
BridgeJobTechnology rows: 66

===== CAPABILITY =====
DimCapability rows: 9
Unique capability IDs: 9
BridgeJobCapability rows: 178

===== SEARCH TERMS =====
DimSearchTerm rows: 3
Unique search-term IDs: 3
BridgeJobSearchTerm rows: 258

===== DATE =====
DimDate rows: 1197
Fact dates missing from DimDate: 0


In [105]:
# Fact table
assert len(FactJobs) == 244
assert FactJobs["job_id"].is_unique

# Dimensions
assert DimTechnology["technology_id"].is_unique
assert DimCapability["capability_id"].is_unique
assert DimSearchTerm["search_term_id"].is_unique
assert DimDate["date"].is_unique

# Bridge uniqueness
assert (
    BridgeJobTechnology[
        ["job_id", "technology_id"]
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    BridgeJobCapability[
        ["job_id", "capability_id"]
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    BridgeJobSearchTerm[
        ["job_id", "search_term_id"]
    ]
    .duplicated()
    .sum()
    == 0
)

# Bridge → FactJobs
assert set(
    BridgeJobTechnology["job_id"]
).issubset(
    set(FactJobs["job_id"])
)

assert set(
    BridgeJobCapability["job_id"]
).issubset(
    set(FactJobs["job_id"])
)

assert set(
    BridgeJobSearchTerm["job_id"]
).issubset(
    set(FactJobs["job_id"])
)

# Bridge → Dimensions
assert set(
    BridgeJobTechnology["technology_id"]
).issubset(
    set(DimTechnology["technology_id"])
)

assert set(
    BridgeJobCapability["capability_id"]
).issubset(
    set(DimCapability["capability_id"])
)

assert set(
    BridgeJobSearchTerm["search_term_id"]
).issubset(
    set(DimSearchTerm["search_term_id"])
)

# Date coverage
assert (
    ~FactJobs["created_date"]
    .isin(DimDate["date"])
).sum() == 0

print("Power BI model validation passed.")

Power BI model validation passed.


In [106]:
print("Power BI files:")

for file in sorted(
    os.listdir(POWERBI_PATH)
):
    print(file)

Power BI files:
BridgeJobCapability.csv
BridgeJobSearchTerm.csv
BridgeJobTechnology.csv
DimCapability.csv
DimDate.csv
DimSearchTerm.csv
DimTechnology.csv
FactJobs.csv
